In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

### Setup
- ready to run for a prefect run with end_WY=2025 and start_WY=2024

In [ ]:
spark = create_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# define POR for recent test
wy = 2025
end_date = pd.to_datetime(f'09-30-{wy}')

### Check primary

In [ ]:
obs_df = ev.primary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()

obs_df.head()

In [ ]:
print(obs_df.shape)

In [ ]:
var_list = obs_df['variable_name'].unique()
print(var_list)

### Check Secondary

In [ ]:
fcst_df = ev.secondary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()
fcst_df.head()

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

### Perform plotting QA/QC

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re


def plot_nrcs_location_wateryear(
    obs_df: pd.DataFrame,
    forecast_df: pd.DataFrame,
    location_id: str,
    water_year: int,
    figsize: tuple = (16, 12),
) -> plt.Figure:
    """Plot observed and forecast data for a single location and water year.
    
    Parameters
    ----------
    obs_df : pd.DataFrame
        Observations dataframe from warehouse with columns: location_id, value_time,
        variable_name, value, etc.
    forecast_df : pd.DataFrame
        Forecasts dataframe from warehouse with columns: location_id, reference_time,
        value_time, variable_name, value, member, etc.
    location_id : str
        The location ID to plot (e.g., 'nrcs-06054500:MT:USGS')
    water_year : int
        Water year to plot (e.g., 2025 means Oct 1, 2024 - Sep 30, 2025)
    figsize : tuple
        Figure size (width, height)
    
    Returns
    -------
    plt.Figure
        Matplotlib figure object
    """
    
    # Water year boundaries
    wy_start = datetime(water_year - 1, 10, 1).date()
    wy_end = datetime(water_year, 9, 30).date()
    
    # Filter observations for this location and water year (by value_time)
    obs_filtered = obs_df[
        (obs_df["location_id"] == location_id) &
        (pd.to_datetime(obs_df["value_time"]).dt.date >= wy_start) &
        (pd.to_datetime(obs_df["value_time"]).dt.date <= wy_end)
    ].copy()
    
    # Filter forecasts for this location and water year (by value_time)
    forecast_filtered = forecast_df[
        (forecast_df["location_id"] == location_id) &
        (pd.to_datetime(forecast_df["value_time"]).dt.date >= wy_start) &
        (pd.to_datetime(forecast_df["value_time"]).dt.date <= wy_end)
    ].copy()
    
    if obs_filtered.empty and forecast_filtered.empty:
        print(f"⚠️ No data found for {location_id} in water year {water_year}")
        return None
    
    # Convert to datetime
    if not obs_filtered.empty:
        obs_filtered["value_time"] = pd.to_datetime(obs_filtered["value_time"])
    if not forecast_filtered.empty:
        forecast_filtered["value_time"] = pd.to_datetime(forecast_filtered["value_time"])
        forecast_filtered["reference_time"] = pd.to_datetime(forecast_filtered["reference_time"])
    
    # Extract forecast periods from variable names
    def extract_period(var_name):
        """Extract (start_month, end_month) from volume_MMDD_sum format."""
        match = re.search(r'volume_(\d{2})(\d{2})', var_name)
        if match:
            return (int(match.group(1)), int(match.group(2)))
        return None
    
    # Get unique forecast periods
    forecast_periods = set()
    if not forecast_filtered.empty:
        for var_name in forecast_filtered["variable_name"].unique():
            period = extract_period(var_name)
            if period:
                forecast_periods.add(period)
    if not obs_filtered.empty:
        for var_name in obs_filtered["variable_name"].unique():
            if "_cumsum" in var_name:
                period = extract_period(var_name)
                if period:
                    forecast_periods.add(period)
    
    forecast_periods = sorted(list(forecast_periods))
    
    # Create figure with dynamic number of subplots
    n_forecast_plots = len(forecast_periods)
    n_rows = 2 + n_forecast_plots  # 2 for monthly + cumsum, then 1 per forecast period
    
    fig, axes = plt.subplots(n_rows, 1, figsize=(figsize[0], figsize[1] + 4 * n_forecast_plots))
    fig.suptitle(f"{location_id} - Water Year {water_year} (Oct {water_year-1} - Sep {water_year})", 
                 fontsize=16, fontweight='bold')
    
    if n_rows == 1:
        axes = [axes]
    
    # --- Plot 0: Monthly Observations ---
    ax = axes[0]
    monthly_obs = obs_filtered[obs_filtered["variable_name"] == "volume_monthly_sum"]
    if not monthly_obs.empty:
        monthly_obs_sorted = monthly_obs.sort_values("value_time")
        ax.plot(monthly_obs_sorted["value_time"], monthly_obs_sorted["value"], 
                marker='o', linewidth=2, markersize=6, label="Monthly Observed", color='blue')
        ax.set_ylabel("Volume (m³)", fontsize=11)
        ax.set_title("Monthly Observations (by observation date)", fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No monthly observations", ha='center', va='center', 
                transform=ax.transAxes, fontsize=11, color='gray')
    
    # --- Plot 1: Cumulative Sum + Final Observed Values ---
    ax = axes[1]
    
    # Plot cumsum (by value_time - observation date)
    cumsum_vars = obs_filtered[obs_filtered["variable_name"].str.contains("_cumsum", na=False)]
    if not cumsum_vars.empty:
        for var_name in cumsum_vars["variable_name"].unique():
            var_data = cumsum_vars[cumsum_vars["variable_name"] == var_name].sort_values("value_time")
            ax.plot(var_data["value_time"], var_data["value"], marker='s', linewidth=2, 
                   markersize=5, label=var_name, alpha=0.8)
    
    # Plot final observed on same axis (by value_time - observation date)
    final_obs = obs_filtered[
        (obs_filtered["variable_name"].str.contains("_sum", na=False)) &
        (~obs_filtered["variable_name"].str.contains("monthly", na=False))
    ]
    if not final_obs.empty:
        for var_name in final_obs["variable_name"].unique():
            var_data = final_obs[final_obs["variable_name"] == var_name].sort_values("value_time")
            ax.scatter(var_data["value_time"], var_data["value"], s=120, 
                      label=f"{var_name} (final)", alpha=0.8, edgecolors='black', linewidth=2, marker='^')
    
    ax.set_ylabel("Volume (m³)", fontsize=11)
    ax.set_title("Cumulative Sum Observations & Final Observed Values (by observation date)", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='best')
    
    # --- Plots 2+: One per Forecast Period (using reference_time for forecasts) ---
    for idx, period in enumerate(forecast_periods):
        ax = axes[2 + idx]
        start_month, end_month = period
        period_var = f"volume_{start_month:02d}{end_month:02d}_sum"
        
        # Plot forecasts and members for this period (by reference_time - forecast issue date)
        period_forecasts = forecast_filtered[
            forecast_filtered["variable_name"] == period_var
        ]
        
        if not period_forecasts.empty:
            # Plot members as faint lines
            for member in period_forecasts["member"].unique():
                member_data = period_forecasts[period_forecasts["member"] == member].sort_values("reference_time")
                ax.plot(member_data["reference_time"], member_data["value"], 
                       alpha=0.15, linewidth=0.8, color='navy')
            
            # Plot mean as bold line
            mean_data = period_forecasts.groupby("reference_time")["value"].mean().reset_index().sort_values("reference_time")
            ax.plot(mean_data["reference_time"], mean_data["value"], marker='d', linewidth=2.5, 
                   markersize=7, label=f"{period_var} forecast (mean)", alpha=0.95, color='navy')
        
        # Plot cumsum for this period (by value_time - observation date)
        cumsum_var = f"volume_{start_month:02d}{end_month:02d}_cumsum"
        period_cumsum = obs_filtered[obs_filtered["variable_name"] == cumsum_var]
        if not period_cumsum.empty:
            cumsum_sorted = period_cumsum.sort_values("value_time")
            ax.plot(cumsum_sorted["value_time"], cumsum_sorted["value"], marker='s', linewidth=2.5, 
                   markersize=6, label=cumsum_var, alpha=0.8, color='green')
        
        # Plot final observed for this period (by value_time - observation date)
        period_final = obs_filtered[obs_filtered["variable_name"] == period_var]
        if not period_final.empty:
            ax.scatter(period_final["value_time"], period_final["value"], s=150, 
                      label=f"{period_var} (final obs)", alpha=0.9, edgecolors='darkred', 
                      linewidth=2.5, marker='^', color='red')
        
        ax.set_ylabel("Volume (m³)", fontsize=11)
        ax.set_title(f"Forecast Period {start_month:02d}-{end_month:02d} (Forecasts by issue date, Obs by observation date)", 
                    fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='best')
    
    # Format x-axis
    for ax in axes:
        ax.tick_params(axis='x', rotation=45)
        ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d"))
    
    plt.tight_layout()
    return fig

#### First WY

In [ ]:
fig = plot_nrcs_location_wateryear(
    obs_df, fcst_df, 
    location_id="nrcs-06054500:MT:USGS",
    water_year=wy-1
)
plt.show()

#### Second WY

In [ ]:
fig = plot_nrcs_location_wateryear(
    obs_df, fcst_df, 
    location_id="nrcs-06054500:MT:USGS",
    water_year=wy
)
plt.show()

### Check dual forecast period site

In [ ]:
fcst_df = ev.secondary_timeseries.filter([
    "location_id = 'nrcs-06054500:MT:USGS'",
    f"value_time < '{end_date}'",
    "member = 'mean'",
]).order_by(
    'value_time'
).to_pandas()

fcst_df.tail(10)

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

### Kill spark

In [ ]:
ev.spark.stop()